# True-LoRA: matutake にコーディング能力を付与するチュートリアル

**Target Model:** [summerMC/matutake](https://huggingface.co/summerMC/matutake) (2B params, Qwen2)

**Hardware:** Colab Free Tier (T4 GPU, ~15GB VRAM)

### 流れ
1. 環境セットアップ
2. アダプタバンク構築
3. True-LoRA トレーニング
4. LoRAアダプタ生成
5. ベースモデルにマージ
6. テスト

---
## Step 1: 環境セットアップ

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
!pip install git+https://github.com/MARVserver/TrueLora.git -q
!pip install transformers accelerate -q
print("Done!")

In [ ]:
import gc, json
from pathlib import Path

import torch
from true_lora.adapter import (
    AdapterBank, AdapterSpec, LoraTensorSpec,
    save_peft_adapter, adapter_fingerprint,
)
from true_lora.generator import TrueLoraGenerator
from true_lora.text import HashingTextEncoder
from true_lora.train import train_on_adapter_bank
from true_lora.reporting import write_json_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def report_gpu():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e6
        reserved = torch.cuda.memory_reserved() / 1e6
        print(f"  GPU: {used:.0f}MB allocated, {reserved:.0f}MB reserved")

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print(f"Device: {DEVICE}")

---
## Step 2: アダプタバンク構築

Colab Free Tier向けに軽量構成:
- **LoRA Rank:** 4（軽量）
- **対象レイヤー:** 4層（0, 6, 12, 18）
- **LoRAターゲット:** attention のみ（q_proj, v_proj）
- **アダプタ数:** 15

In [ ]:
# 軽量パラメータ
TEXT_DIM = 128
HIDDEN_DIM = 256
LORA_RANK = 4
LORA_ALPHA = 8.0
MAX_NORM = 4.0

# Qwen2: attention のみ（メモリ節約）
LORA_TARGETS = ["q_proj", "v_proj"]
TARGET_LAYERS = [0, 6, 12, 18]  # 4層のみ

# 隠れ層サイズを動的取得
from transformers import AutoConfig
config = AutoConfig.from_pretrained("summerMC/matutake", trust_remote_code=True)
HIDDEN_SIZE = config.hidden_size

print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Target layers: {TARGET_LAYERS}")
print(f"LoRA targets: {LORA_TARGETS}")
print(f"Total specs: {len(TARGET_LAYERS) * len(LORA_TARGETS)}")

In [ ]:
# LoRA TensorSpecを生成
lora_specs = []
for layer_idx in TARGET_LAYERS:
    for target in LORA_TARGETS:
        name = f"model.layers.{layer_idx}.self_attn.{target}"
        lora_specs.append(LoraTensorSpec(
            name=name,
            out_features=HIDDEN_SIZE,
            in_features=HIDDEN_SIZE,
            rank=LORA_RANK,
            alpha=LORA_ALPHA,
        ))

print(f"LoRA specs: {len(lora_specs)}")
for s in lora_specs:
    print(f"  {s.name}: A={s.a_shape}, B={s.b_shape}")

In [ ]:
# アダプタバンクを作成（15個に削減）
encoder = HashingTextEncoder(dim=TEXT_DIM)

coding_adapters = [
    {"desc": "python code generation", "sa": 0.15, "sb": 0.08, "score": 0.85},
    {"desc": "python function implementation", "sa": 0.18, "sb": 0.09, "score": 0.82},
    {"desc": "algorithm implementation", "sa": 0.20, "sb": 0.10, "score": 0.88},
    {"desc": "data structure implementation", "sa": 0.17, "sb": 0.08, "score": 0.83},
    {"desc": "dynamic programming solutions", "sa": 0.22, "sb": 0.11, "score": 0.86},
    {"desc": "code debugging and error fixing", "sa": 0.13, "sb": 0.06, "score": 0.79},
    {"desc": "unit test writing with pytest", "sa": 0.15, "sb": 0.07, "score": 0.77},
    {"desc": "performance optimization", "sa": 0.18, "sb": 0.09, "score": 0.80},
    {"desc": "numpy array operations", "sa": 0.16, "sb": 0.08, "score": 0.82},
    {"desc": "torch deep learning code", "sa": 0.19, "sb": 0.09, "score": 0.85},
    {"desc": "design pattern implementation", "sa": 0.17, "sb": 0.08, "score": 0.78},
    {"desc": "object oriented programming", "sa": 0.15, "sb": 0.07, "score": 0.76},
    {"desc": "type hints and annotations", "sa": 0.11, "sb": 0.05, "score": 0.72},
    {"desc": "file io and pathlib usage", "sa": 0.10, "sb": 0.05, "score": 0.71},
    {"desc": "code refactoring and cleanup", "sa": 0.12, "sb": 0.06, "score": 0.74},
]

adapters = []
for cfg in coding_adapters:
    tensors = {}
    for spec in lora_specs:
        tensors[f"{spec.name}.lora_A.weight"] = torch.full(spec.a_shape, cfg["sa"])
        tensors[f"{spec.name}.lora_B.weight"] = torch.full(spec.b_shape, cfg["sb"])
    adapters.append(AdapterSpec(
        description=cfg["desc"],
        embedding=encoder.encode(cfg["desc"]),
        tensors=tensors,
        metrics={"score": cfg["score"]},
    ))

bank = AdapterBank(adapters)
print(f"Adapter bank: {len(bank.adapters)} adapters")
report_gpu()

---
## Step 3: True-LoRA トレーニング

In [ ]:
# 生成器を作成
generator = TrueLoraGenerator(
    tensor_specs=lora_specs,
    adapter_bank=bank,
    text_dim=TEXT_DIM,
    hidden_dim=HIDDEN_DIM,
    max_tensor_norm=MAX_NORM,
)

params = sum(p.numel() for p in generator.hyper.parameters())
print(f"HyperAdapter params: {params:,}")
report_gpu()

In [ ]:
# トレーニング
print("Training...")
losses = train_on_adapter_bank(generator, bank.adapters, steps=200, lr=1e-3)
print(f"Done! Final loss: {losses[-1]:.6f}")
report_gpu()

---
## Step 4: LoRAアダプタの生成

In [ ]:
# LoRAを生成
prompt = "python code generation"
state_dict, report = generator.generate(prompt, retrieval_k=8)

print(f"Uncertainty: {report['uncertainty']:.4f}")
print(f"Max retrieval score: {report['max_retrieval_score']:.4f}")
print(f"Retrieved adapters:")
for p in report["retrieved_adapters"]:
    print(f"  [{p['rank']}] {p['description']} (w={p['weight']:.3f})")
print(f"Tensor count: {len(state_dict)}")

In [ ]:
# 保存
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

adapter_path = OUTPUT_DIR / "coding_lora.pt"
save_peft_adapter(adapter_path, state_dict, report)
print(f"Saved: {adapter_path}")

report_path = OUTPUT_DIR / "generation_report.json"
write_json_report(report_path, report)
print(f"Report: {report_path}")

---
## Step 5: マージ

生成器を削除してメモリを解放してからマージする。

In [ ]:
# メモリ解放
del generator, bank, adapters, state_dict, losses
clear_gpu()
print("Memory cleared")
report_gpu()

In [ ]:
from true_lora.merge import merge_adapter_into_hf_model

merged_dir = OUTPUT_DIR / "matutake_coding_merged"
print(f"Merging into summerMC/matutake...")
print("(This downloads ~4GB model, takes ~5-10 min)\n")

merge_report = merge_adapter_into_hf_model(
    adapter_path=adapter_path,
    model_name_or_path="summerMC/matutake",
    output_dir=merged_dir,
    dtype="bfloat16",
    device="cpu",
    allow_download=True,
)

print(f"\nDone!")
print(f"Output: {merge_report['output_dir']}")
print(f"Applied: {merge_report['applied_layers']}")

---
## Step 6: テスト

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading merged model...")
tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Loaded! {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

In [ ]:
def test_coding(user_prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
print("Test 1: Binary Search")
print("=" * 50)
print(test_coding("Write a Python function that implements binary search. Include type hints."))

In [ ]:
print("Test 2: Dynamic Programming")
print("=" * 50)
print(test_coding("Implement longest common subsequence using dynamic programming."))

In [ ]:
print("Test 3: Japanese")
print("=" * 50)
print(test_coding("Pythonでリストの重複を除いた要素を返す関数を書いて。メソッドを使わずに。"))

---
## HuggingFace Hubにアップロード（オプション）

In [ ]:
USE_HF_UPLOAD = False
HF_REPO = "summerMC/matutake-coding-lora"

if USE_HF_UPLOAD:
    from huggingface_hub import login, HfApi
    login()
    HfApi().upload_folder(folder_path=str(merged_dir), repo_id=HF_REPO, repo_type="model")
    print(f"Uploaded: https://huggingface.co/{HF_REPO}")
else:
    print(f"Manual: huggingface-cli upload {HF_REPO} {merged_dir} .")

---
## まとめ

| 項目 | 値 |
| --- | --- |
| LoRA Rank | 4 |
| LoRA Alpha | 8.0 |
| Target Layers | 4 (0, 6, 12, 18) |
| LoRA Targets | q_proj, v_proj |
| Total LoRA Specs | 8 |
| Adapter Bank | 15 adapters |
| Training Steps | 200 |

### 使用例

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "summerMC/matutake-coding-lora",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("summerMC/matutake-coding-lora")

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to sort a list."},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
```